## Append checkpoint11 rows into the existing checkpoint15 final CSV.


This script is intentionally idempotent: rows already present in checkpoint15 are
skipped using a stable incident key. New checkpoint11 rows are aligned to the
checkpoint15 schema before writing, so final-only columns that checkpoint11 does
not have are added as missing values.
"""

In [1]:
from __future__ import annotations

import argparse
import shutil
from datetime import datetime
from pathlib import Path

import pandas as pd


In [10]:
REPO_ROOT = Path.cwd().resolve().parent



SOURCE_PATH = REPO_ROOT / "streamlit-app" / "checkpoint11_combined_data.csv"
TARGET_PATH = REPO_ROOT / "streamlit-app" / "checkpoint15_misdemeanor_warrant.csv"

DEDUP_KEY = ["Incident #", "Date", "Type", "person_id"]

DRY_RUN = False  # Change to False when you're ready to actually overwrite checkpoint15

print("Repo root:", REPO_ROOT)
print("Source exists:", SOURCE_PATH.exists(), SOURCE_PATH)
print("Target exists:", TARGET_PATH.exists(), TARGET_PATH)

Repo root: /Users/sreej/Desktop/Gateway_lawrence/gatewayinitiative-lawrencepd
Source exists: True /Users/sreej/Desktop/Gateway_lawrence/gatewayinitiative-lawrencepd/streamlit-app/checkpoint11_combined_data.csv
Target exists: True /Users/sreej/Desktop/Gateway_lawrence/gatewayinitiative-lawrencepd/streamlit-app/checkpoint15_misdemeanor_warrant.csv


In [11]:
def normalize_key_part(series):
    return series.fillna("").astype(str).str.strip()


def build_row_key(df, key_columns):
    missing = [column for column in key_columns if column not in df.columns]
    if missing:
        raise ValueError(f"Missing dedupe key columns: {missing}")

    key = normalize_key_part(df[key_columns[0]])
    for column in key_columns[1:]:
        key = key + "||" + normalize_key_part(df[column])

    return key


def align_to_target_schema(source_df, target_columns):
    aligned = source_df.copy()

    for column in target_columns:
        if column not in aligned.columns:
            aligned[column] = pd.NA

    return aligned[target_columns]


def append_checkpoint(source_path, target_path, dry_run=True):
    if not source_path.exists():
        raise FileNotFoundError(f"Source CSV not found: {source_path}")

    if not target_path.exists():
        raise FileNotFoundError(f"Target CSV not found: {target_path}")

    source_df = pd.read_csv(source_path, low_memory=False)
    target_df = pd.read_csv(target_path, low_memory=False)

    source_aligned = align_to_target_schema(source_df, list(target_df.columns))

    existing_keys = set(build_row_key(target_df, DEDUP_KEY))
    source_keys = build_row_key(source_aligned, DEDUP_KEY)

    new_rows = source_aligned.loc[~source_keys.isin(existing_keys)].copy()

    new_rows_for_concat = new_rows.dropna(axis=1, how="all")
    combined_df = pd.concat([target_df, new_rows_for_concat], ignore_index=True)

    print(f"Source rows:       {len(source_df):,}")
    print(f"Existing rows:     {len(target_df):,}")
    print(f"Rows to append:    {len(new_rows):,}")
    print(f"Final row count:   {len(combined_df):,}")
    print(f"Target columns:    {len(target_df.columns):,}")

    if dry_run:
        print("Dry run only. No files were changed.")
        return combined_df, new_rows

    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    backup_path = target_path.with_name(
        f"{target_path.stem}.backup_{timestamp}{target_path.suffix}"
    )

    shutil.copy2(target_path, backup_path)
    combined_df.to_csv(target_path, index=False)

    print(f"Backup written:    {backup_path}")
    print(f"Updated CSV:       {target_path}")

    return combined_df, new_rows

In [12]:
combined_df, new_rows = append_checkpoint(
    source_path=SOURCE_PATH,
    target_path=TARGET_PATH,
    dry_run=DRY_RUN,
)

Source rows:       66,741
Existing rows:     428,527
Rows to append:    66,595
Final row count:   495,122
Target columns:    21
Backup written:    /Users/sreej/Desktop/Gateway_lawrence/gatewayinitiative-lawrencepd/streamlit-app/checkpoint15_misdemeanor_warrant.backup_20261005_130823.csv
Updated CSV:       /Users/sreej/Desktop/Gateway_lawrence/gatewayinitiative-lawrencepd/streamlit-app/checkpoint15_misdemeanor_warrant.csv
